# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2) dataset using the `mlcroissant` library. 

It demonstrates how to use Croissant's schema to:
- Load metadata and records
- Explore the structure (record sets, fields, columns) via their `@id`s
- Extract and manipulate tabular data
- Perform exploratory data analysis (EDA)
- Visualize the results

## Dataset Source
The Croissant schema for this dataset is available at:
- https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and records using `mlcroissant`. The dataset is described by a Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant metadata URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print essential metadata fields (name & description)
print(f"Dataset loaded: {dataset.metadata.name}\n{dataset.metadata.description}\n")

## 2. Data Overview

Review available record sets, including their `@id`s and their fields/columns. This step is important for understanding the dataset schema and for referencing components by their unique Croissant `@id` identifiers.

**Note:** In Croissant, each Record Set (table or entity) has a unique `@id`, and each field/column is referenced by an `@id` inside each record set. You should use these `@id`s for accessing data, not simple field names.

In [ ]:
# List all available RecordSet @id's with their field/column @id's
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} Record Sets\n")
for rset in record_sets:
    print(f"RecordSet @id: {rset['@id']}")
    if 'field' in rset:
        field_ids = []
        if isinstance(rset['field'], list):
            field_ids = [f['@id'] if isinstance(f, dict) else f for f in rset['field']]
        else:
            field_ids = [rset['field']['@id'] if isinstance(rset['field'], dict) else rset['field']]
        print(f"    Fields: {field_ids}")
    if 'column' in rset:
        col_ids = []
        if isinstance(rset['column'], list):
            col_ids = [c['@id'] if isinstance(c, dict) else c for c in rset['column']]
        else:
            col_ids = [rset['column']['@id'] if isinstance(rset['column'], dict) else rset['column']]
        print(f"    Columns: {col_ids}")
    print()
# Store all the RecordSet @ids for later use
record_set_ids = [rset['@id'] for rset in record_sets]
if len(record_set_ids) == 0:
    print("No record sets detected in the dataset. Please check schema structure.")

## 3. Data Extraction

Load records from each record set into Pandas DataFrames, referencing them using their exact `@id`s. Preview columns (`@id`s) for exploration.

**If record sets contain a large amount of data, consider sampling instead of loading all rows.**

In [ ]:
# Load records for each record set by @id, using Croissant's `records` API
dataframes = {}
for rset_id in record_set_ids:
    print(f"Loading records for RecordSet: {rset_id}")
    # Records is a generator of dicts keyed by field/column @id
    records = list(dataset.records(record_set=rset_id))
    df = pd.DataFrame(records)
    dataframes[rset_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Sample records:\n{df.head(2)}\n")

# For demonstration, select the first available RecordSet
if len(record_set_ids) > 0:
    main_rset_id = record_set_ids[0]
    print(f"Defaulting to first RecordSet for analysis: {main_rset_id}")
    main_df = dataframes[main_rset_id]
else:
    main_rset_id = None
    main_df = pd.DataFrame()

# List the columns available in main_df (by Croissant @id)
print(f"Columns available for {main_rset_id}: {main_df.columns.tolist()}")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps such as:
- Filtering records by a numeric field (using that field's `@id`)
- Normalizing numeric fields
- Grouping/summarizing by categorical attribute

All references to columns/fields below use their Croissant `@id` values from the previous overview, ensuring stable access across versions.

In [ ]:
# Example: Pick a numeric field for analysis. Adjust the field @id if needed.
# For demonstration, we select first numeric dtype field from main_rset_id

numeric_field_id = None
if len(main_df.columns) > 0:
    for col in main_df.columns:
        if np.issubdtype(main_df[col].dtype, np.number):
            numeric_field_id = col
            break

if numeric_field_id is None:
    # If all dtypes are object, try to force numeric
    for col in main_df.columns:
        coerced = pd.to_numeric(main_df[col], errors='coerce')
        if coerced.notnull().sum() > 0:
            main_df[col] = coerced
            if np.issubdtype(main_df[col].dtype, np.number):
                numeric_field_id = col
                break

if numeric_field_id is not None:
    print(f"Selected numeric field for analysis (by @id): {numeric_field_id}")
    threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].notnull().sum() else 0
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
    print(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nFirst 5 normalized {numeric_field_id} values:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Attempt to select a second field to group by
    group_field_id = None
    for col in main_df.columns:
        if col != numeric_field_id and main_df[col].dtype == object:
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped average {numeric_field_id} by {group_field_id} (by @id):")
        print(grouped_df.head())
else:
    print("No numeric field detected for analysis.")

## 5. Visualization

Visualize the distribution of the selected numeric field and its relationship with the group field, using matplotlib and seaborn for illustration.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and filtered_df.shape[0] > 0:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Grouped barplot, if a grouping field is available
    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10, 6))
        group_means = filtered_df.groupby(group_field_id)[numeric_field_id].mean().sort_values().reset_index()
        sns.barplot(data=group_means, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} per {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean of {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

This notebook demonstrated how to use the Croissant schema with `mlcroissant` for a reproducible, standards-compliant workflow in dataset exploration:
- All analysis references fields, columns, and record sets by their unique `@id`.
- Data discovery, extraction, transformation, and visualization are performed in a composable pipeline.

You can extend this workflow for downstream ML tasks or further FAIR analysis following the Croissant data contract and best practices.

_For more: visit [mlcroissant on PyPI](https://pypi.org/project/mlcroissant/) or view [open Croissant schemas](https://mlcommons.org/datadriven/croissant/)_